# Factorial (C++)

[![Apri in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Factorial_cpp.ipynb)

Capitolo 14 — Ricorsione. Notebook eseguibile di `Factorial`: l'algoritmo in C++, eseguito sugli esempi del capitolo, con la cella «Prova tu» per provarlo con dati tuoi. Lo pseudocodice, gli esempi e le dimostrazioni sono nella [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

Lo stesso notebook in [Python](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Factorial_python.ipynb) e [Java](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione/Factorial_java.ipynb); tutti gli algoritmi del capitolo nel [notebook del capitolo](https://colab.research.google.com/github/fabiofurini/algoritmi/blob/main/notebooks/cap14_ricorsione_cpp.ipynb).

**Come si esegue.** Clic sul badge «Apri in Colab», poi *Runtime → Esegui tutto* (Ctrl+F9), oppure una cella alla volta con Maiusc+Invio, dall'alto in basso. Ogni programma C++ è scritto su un file dalla cella `%%writefile`, poi compilato con `g++` ed eseguito nella cella successiva: tutto avviene in Colab.

**Nomi.** In questo capitolo non ci sono array: variabili e parametri hanno gli stessi nomi dello pseudocodice. Lo stack e i contatori sono variabili globali, e la traccia si attiva con `trace = true`.

## Fattoriale: Factorial

**Input:** un numero naturale $n$. **Output:** $n!$, con $0! = 1$. $S(n) = 2n + 2 = \Theta(n)$ passi; $n + 1$ chiamate, profondità $n + 1$ (l'albero è un cammino).

Pseudocodice: sezione 14.4 della [pagina del capitolo](https://fabiofurini.github.io/algoritmi/ricorsione/14-ricorsione/).

In [1]:
%%writefile factorial.cpp
#include <cstdarg>
#include <cstdio>
#include <cmath>
#include <initializer_list>
#include <string>
using namespace std;

long long cnt[20];      // cnt[r] = numero di esecuzioni della riga r (numerazione delle dispense)
long long calls = 0;    // numero di chiamate = nodi dell'albero delle chiamate
int maxDepth = 0;       // profondita' della ricorsione = altezza massima dello stack
long long mults = 0;    // moltiplicazioni (FastPower)
long long moves = 0;    // mosse (Hanoi)
int ev = 0;             // contatore degli eventi della traccia
int stk[1000];          // lo stack delle chiamate: gli argomenti delle chiamate attive (al massimo 1000)
int sp = 0;             // numero di chiamate attive
long long seen[64];     // seen[x] = numero di chiamate con argomento x
bool trace = false;     // true: stampa la traccia dell'esecuzione

void Reset()
{
    for (int r = 0; r < 20; r++) cnt[r] = 0;
    for (int k = 0; k < 64; k++) seen[k] = 0;
    calls = 0; maxDepth = 0; mults = 0; moves = 0; ev = 0; sp = 0;
}

long long Total()
{
    long long s = 0;
    for (int r = 0; r < 20; r++) s += cnt[r];
    return s;
}

void PrintSteps(initializer_list<int> lines)
{
    printf("Riga | Esecuzioni\n");
    for (int r : lines) printf("%4d | %10lld\n", r, cnt[r]);
    printf("Passi totali S = %lld\n", Total());
}

string Str(const char* fmt, ...)       // come sprintf, ma restituisce una stringa
{
    char buf[256];
    va_list ap;
    va_start(ap, fmt);
    vsnprintf(buf, sizeof buf, fmt, ap);
    va_end(ap);
    return string(buf);
}

void Push(int x)                       // inizia una chiamata: il suo frame va in cima allo stack
{
    calls++;
    stk[sp++] = x;
    if (sp > maxDepth) maxDepth = sp;
    if (x >= 0 && x < 64) seen[x]++;
}

void Pop()                             // la chiamata termina: il suo frame esce dallo stack
{
    sp--;
}

string StackStr()                      // "[4, 3, 2]"
{
    string s = "[";
    for (int k = 0; k < sp; k++)
    {
        if (k > 0) s += ", ";
        s += to_string(stk[k]);
    }
    return s + "]";
}

void Event(const string& text)         // riga numerata della traccia, con lo stack
{
    ev++;
    printf("%2d. %s   stack: %s\n", ev, text.c_str(), StackStr().c_str());
}

void Say(const string& call, const string& action)   // riga della traccia: la chiamata, rientrata di un livello per ogni chiamata attiva
{
    printf("%-28s| %s\n", (string(2 * (sp - 1), ' ') + call).c_str(), action.c_str());
}

// Factorial(n) valori corretti solo fino a n = 20 (in C++ e Java i 64 bit traboccano oltre)
unsigned long long Factorial(int n)
{
    Push(n); cnt[2]++;
    if (n == 0)
    {
        cnt[3]++;
        if (trace) Event("Factorial(0): n = 0 => caso base");
        Pop();
        if (trace) Event("Factorial(0) restituisce 1");
        return 1;
    }
    if (trace) Event(Str("Factorial(%d): %d != 0 => deve calcolare %d * Factorial(%d) e attende", n, n, n, n - 1));
    cnt[5]++;
    unsigned long long f = Factorial(n - 1);
    unsigned long long r = n * f;
    Pop();
    if (trace) Event(Str("Factorial(%d) restituisce %d * %llu = %llu", n, n, f, r));
    return r;
}

#ifndef SOLO_LIBRERIA
int main()
{
    // Esempio delle dispense: Factorial(4), con lo stack
    int n = 4;
    Reset();
    trace = true;
    unsigned long long r = Factorial(n);
    trace = false;
    printf("Factorial(%d) = %llu\n", n, r);
    printf("Chiamate = %lld (nodi dell'albero), profondita' della ricorsione = %d\n", calls, maxDepth);
    PrintSteps({2, 3, 5});
    printf("S(n) = 2n + 2 = %d\n", 2 * n + 2);

    // Chiamate, profondita' e passi al variare di n
    printf("\n");
    printf("  n |                   n! | chiamate | profondita' | S(n) | 2n + 2\n");
    int values[] = {0, 1, 2, 3, 4, 5, 10, 15, 20};
    for (int v : values)
    {
        Reset();
        unsigned long long f = Factorial(v);
        printf("%3d | %20llu | %8lld | %11d | %4lld | %d\n", v, f, calls, maxDepth, Total(), 2 * v + 2);
    }
    return 0;
}
#endif

Writing factorial.cpp


In [2]:
!g++ -std=c++17 -O2 -o factorial factorial.cpp && ./factorial

 1. Factorial(4): 4 != 0 => deve calcolare 4 * Factorial(3) e attende   stack: [4]
 2. Factorial(3): 3 != 0 => deve calcolare 3 * Factorial(2) e attende   stack: [4, 3]
 3. Factorial(2): 2 != 0 => deve calcolare 2 * Factorial(1) e attende   stack: [4, 3, 2]
 4. Factorial(1): 1 != 0 => deve calcolare 1 * Factorial(0) e attende   stack: [4, 3, 2, 1]
 5. Factorial(0): n = 0 => caso base   stack: [4, 3, 2, 1, 0]
 6. Factorial(0) restituisce 1   stack: [4, 3, 2, 1]
 7. Factorial(1) restituisce 1 * 1 = 1   stack: [4, 3, 2]
 8. Factorial(2) restituisce 2 * 1 = 2   stack: [4, 3]
 9. Factorial(3) restituisce 3 * 2 = 6   stack: [4]
10. Factorial(4) restituisce 4 * 6 = 24   stack: []
Factorial(4) = 24
Chiamate = 5 (nodi dell'albero), profondita' della ricorsione = 5
Riga | Esecuzioni
   2 |          5
   3 |          1
   5 |          4
Passi totali S = 10
S(n) = 2n + 2 = 10

  n |                   n! | chiamate | profondita' | S(n) | 2n + 2
  0 |                    1 |        1 |           1 | 

**Prova tu.** Cambia $n$ e guarda lo stack crescere e poi svuotarsi.

In [3]:
%%writefile prova_factorial.cpp
#define SOLO_LIBRERIA   // non eseguire il main di factorial.cpp
#include "factorial.cpp"

int main()
{
    int n = 6;        // <-- cambia n (al massimo 20)
    Reset();
    trace = true;
    unsigned long long r = Factorial(n);
    trace = false;
    printf("Factorial(%d) = %llu, chiamate = %lld, profondita' = %d, S = %lld\n", n, r, calls, maxDepth, Total());
    return 0;
}

Writing prova_factorial.cpp


In [4]:
!g++ -std=c++17 -O2 -o prova_factorial prova_factorial.cpp && ./prova_factorial

 1. Factorial(6): 6 != 0 => deve calcolare 6 * Factorial(5) e attende   stack: [6]
 2. Factorial(5): 5 != 0 => deve calcolare 5 * Factorial(4) e attende   stack: [6, 5]
 3. Factorial(4): 4 != 0 => deve calcolare 4 * Factorial(3) e attende   stack: [6, 5, 4]
 4. Factorial(3): 3 != 0 => deve calcolare 3 * Factorial(2) e attende   stack: [6, 5, 4, 3]
 5. Factorial(2): 2 != 0 => deve calcolare 2 * Factorial(1) e attende   stack: [6, 5, 4, 3, 2]
 6. Factorial(1): 1 != 0 => deve calcolare 1 * Factorial(0) e attende   stack: [6, 5, 4, 3, 2, 1]
 7. Factorial(0): n = 0 => caso base   stack: [6, 5, 4, 3, 2, 1, 0]
 8. Factorial(0) restituisce 1   stack: [6, 5, 4, 3, 2, 1]
 9. Factorial(1) restituisce 1 * 1 = 1   stack: [6, 5, 4, 3, 2]
10. Factorial(2) restituisce 2 * 1 = 2   stack: [6, 5, 4, 3]
11. Factorial(3) restituisce 3 * 2 = 6   stack: [6, 5, 4]
12. Factorial(4) restituisce 4 * 6 = 24   stack: [6, 5]
13. Factorial(5) restituisce 5 * 24 = 120   stack: [6]
14. Factorial(6) restituisce 6 * 120

---

Materiale didattico di [Fabio Furini](https://fabiofurini.github.io/) — DIAG, Sapienza Università di Roma.
Testo, figure e dati [CC BY 4.0](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE), codice [MIT](https://github.com/fabiofurini/algoritmi/blob/main/LICENSE-CODE).